In [43]:
print("Linear Regression Project")

Linear Regression Project


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)

df = data.frame   # this includes the target

In [ ]:
display(df.head())

In [ ]:
!pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

# Generate the profile report
profile = ProfileReport(df, title="Pandas Profiling Report")

# Display the report
profile.to_notebook_iframe()

In [ ]:
# Save the report to an HTML file
profile.to_file("california_housing_profile_report.html")

The report has been saved as `california_housing_profile_report.html`. You can download it from the file browser on the left-hand side of Colab (look for the folder icon, then find the file in the `/content/` directory).

To convert this HTML file to a PDF, follow these steps:

1.  Open the downloaded `california_housing_profile_report.html` file in your web browser.
2.  Use your browser's print function (usually `Ctrl+P` or `Cmd+P`).
3.  In the print dialog, select 'Save as PDF' or 'Print to PDF' as the destination.
4.  Save the PDF to your desired location.

In [ ]:
print(df.columns)

In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
sns.histplot(y, kde=True)

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get all column names from the DataFrame
columns = df.columns

# Determine the number of rows and columns for the subplot grid
# For example, if you have 9 columns, 3 rows and 3 columns would be good.
# You can adjust these numbers based on the total number of columns in your df.
num_cols = len(columns)
num_rows = (num_cols + 2) // 3 # Roughly calculate rows needed for 3 columns per row

plt.figure(figsize=(18, num_rows * 5)) # Adjust figure size dynamically

for i, col in enumerate(columns):
    plt.subplot(num_rows, 3, i + 1) # Create subplot for each column
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import numpy as np
import pandas as pd

# columns to log-transform
log_cols = ["AveRooms", "AveBedrms", "AveOccup", "Population"]

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("log", FunctionTransformer(np.log1p), log_cols)
    ],
    remainder="passthrough"
)

# -----------------------------
# FIT on train → TRANSFORM both
# -----------------------------
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

# -----------------------------
# Rebuild column names
# -----------------------------
remaining_cols = [c for c in X_train.columns if c not in log_cols]
final_cols = log_cols + remaining_cols

# -----------------------------
# Convert to DataFrame
# -----------------------------
X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=final_cols,
    index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=final_cols,
    index=X_test.index
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X["Longitude"],
    y=X["Latitude"],
    hue=y,
    palette="viridis",
    s=10 # Adjust point size for better visualization
)
plt.title("California housing prices by location")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

This scatter plot visualizes the geographical distribution of housing prices across California.

*   The **x-axis** represents `Longitude` (east-west position).
*   The **y-axis** represents `Latitude` (north-south position).
*   The **color** of each point represents the `MedHouseVal` (target variable), with the `viridis` palette typically showing higher values in lighter/yellowish colors and lower values in darker/purple colors.

From this plot, you can infer:
*   **Geographical Concentration**: Are higher or lower values clustered in specific regions?
*   **Trends**: Do prices generally increase or decrease as you move north/south or east/west?
*   **Outliers**: Are there any isolated areas with unusually high or low values compared to their surroundings?

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np

log_cols = ["AveRooms", "AveBedrms", "AveOccup", "Population"]

plt.figure(figsize=(12, 8))

for i, col in enumerate(log_cols):

    # BEFORE log
    plt.subplot(len(log_cols), 2, 2*i + 1)
    stats.probplot(X_train[col], dist="norm", plot=plt)
    plt.title(f"{col} (Original)")

    # AFTER log
    plt.subplot(len(log_cols), 2, 2*i + 2)
    stats.probplot(np.log1p(X_train[col]), dist="norm", plot=plt)
    plt.title(f"{col} (Log Transformed)")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

**Reasoning**:
To fulfill the subtask, I need to import the `StandardScaler` class from `sklearn.preprocessing`. This will be done in a new code cell.



In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = ColumnTransformer(
    [("scale", StandardScaler(), X_train_transformed.columns)],
    remainder="passthrough"
)

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_transformed),
    columns=X_train_transformed.columns,
    index=X_train_transformed.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_transformed),
    columns=X_test_transformed.columns,
    index=X_test_transformed.index
)

In [ ]:
X_train_scaled
X_test_scaled

y_train_log
y_test_log

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train_scaled, y_train_log)

In [ ]:
y_pred_log = lr.predict(X_test_scaled)

In [ ]:
import numpy as np

y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

print("R²:", r2_score(y_test_actual, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred)))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(X_train_scaled.corr(), annot=False, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# -----------------------------
# Drop columns from scaled data
# -----------------------------
X_train_v2 = X_train_scaled.drop(["AveBedrms", "Population"], axis=1)
X_test_v2  = X_test_scaled.drop(["AveBedrms", "Population"], axis=1)

# -----------------------------
# Train model
# -----------------------------
lr_v2 = LinearRegression()
lr_v2.fit(X_train_v2, y_train_log)

# -----------------------------
# Predict
# -----------------------------
y_pred_log_v2 = lr_v2.predict(X_test_v2)

# -----------------------------
# Convert back to original scale
# -----------------------------
y_pred_v2 = np.expm1(y_pred_log_v2)
y_test_actual = np.expm1(y_test_log)

# -----------------------------
# Evaluate
# -----------------------------
print("R²:", r2_score(y_test_actual, y_pred_v2))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred_v2)))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

cv_scores = cross_val_score(
    lr,
    X_train_v2,
    y_train_log,
    cv=5,
    scoring="r2"
)

print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression

param_grid = {
    "fit_intercept": [True, False],
    "positive": [True, False]
}

grid = GridSearchCV(
    LinearRegression(),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train_v2, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

In [ ]:
best_lr = grid.best_estimator_

y_pred_log = best_lr.predict(X_test_v2)
y_pred = np.expm1(y_pred_log)

print("Test R²:", r2_score(y_test_actual, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred)))

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_v2)
X_test_poly  = poly.transform(X_test_v2)


scaler = StandardScaler()

X_train_poly = scaler.fit_transform(X_train_poly)
X_test_poly  = scaler.transform(X_test_poly)

In [ ]:
from sklearn.linear_model import LinearRegression

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train_log)

In [ ]:
y_pred_log_poly = lr_poly.predict(X_test_poly)

In [ ]:
import numpy as np

y_pred_poly = np.expm1(y_pred_log_poly)
y_test_actual = np.expm1(y_test_log)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

print("R²:", r2_score(y_test_actual, y_pred_poly))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred_poly)))

In [ ]:
poly.get_feature_names_out()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

cv_scores_poly = cross_val_score(
    lr,
    X_train_poly,
    y_train_log,
    cv=5,
    scoring="r2"
)

print("Polynomial CV R²:", cv_scores_poly)
print("Mean Polynomial CV R²:", cv_scores_poly.mean())

In [ ]:
residuals = y_test_actual - y_pred_poly

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_pred_poly, residuals)
plt.axhline(0, color="red")
plt.xlabel("Predicted price")
plt.ylabel("Residuals")
plt.title("Residual vs Predicted")
plt.show()

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": poly.get_feature_names_out(), # Use feature names from polynomial transformation
    "Coefficient": lr_poly.coef_             # Use coefficients from the polynomial linear regression model
})

importance["abs_coef"] = importance["Coefficient"].abs()
importance = importance.sort_values("abs_coef", ascending=False)

importance

In [ ]:
importance.sort_values("abs_coef").plot(
    x="Feature",
    y="Coefficient",
    kind="barh",
    figsize=(8,6)
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

degrees = [1, 2, 3, 4]

results = []

for d in degrees:

    model = Pipeline([
        ("scale", StandardScaler()),
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("lr", LinearRegression())
    ])

    scores = cross_val_score(
        model,
        X_train_v2,
        y_train_log,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    results.append({
        "degree": d,
        "mean_cv_r2": scores.mean(),
        "std": scores.std()
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
best_degree = results_df.loc[results_df["mean_cv_r2"].idxmax()]
print("\nBest polynomial degree:")
print(best_degree)

In [ ]:
best_d = int(best_degree["degree"])

final_model = Pipeline([
    ("scale", StandardScaler()),
    ("poly", PolynomialFeatures(degree=best_d, include_bias=False)),
    ("lr", LinearRegression())
])

final_model.fit(X_train_v2, y_train_log)